## 📦 Imports

In [1]:
import pandas as pd
import numpy as np
import pickle
import os
import sys
import time
from itertools import combinations
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
import faiss
from concurrent.futures import ThreadPoolExecutor, as_completed

sys.path.append('../src')
from ingest import normalize_to_ingredient_rxcui

## 📂 Load Knowledge Base

In [2]:
kb_df = pd.read_csv('../data/processed_interactions_kb.csv')
kb_df['pair_key'] = kb_df['pair_key'].apply(eval)

print(f"✅ Loaded knowledge base:")
print(f"   Total interactions: {len(kb_df):,}")
print(f"   Unique drug pairs: {kb_df['pair_key'].nunique():,}")
print(f"\nSample:")
print(kb_df[['Drug 1', 'Drug 2', 'Interaction Description', 'pair_key']].head())

✅ Loaded knowledge base:
   Total interactions: 170,782
   Unique drug pairs: 170,782

Sample:
                Drug 1       Drug 2  \
0           Trioxsalen  Verteporfin   
1  Aminolevulinic acid  Verteporfin   
2     Titanium dioxide  Verteporfin   
3     Tiaprofenic acid  Verteporfin   
4          Cyamemazine  Verteporfin   

                             Interaction Description          pair_key  
0  Trioxsalen may increase the photosensitizing a...   (10844, 118886)  
1  Aminolevulinic acid may increase the photosens...     (118886, 683)  
2  Titanium dioxide may increase the photosensiti...   (118886, 38323)  
3  Tiaprofenic acid may increase the photosensiti...  (118886, 618442)  
4  Cyamemazine may increase the photosensitizing ...   (118886, 21877)  


## 📂 Load RxNorm Mappings

In [3]:
with open('../data/rxnorm_lookups.pkl', 'rb') as f:
    lookups = pickle.load(f)

name_to_rxcui = lookups['name_to_rxcui']
rxcui_to_names = lookups['rxcui_to_names']
bn_to_in_map = lookups['bn_to_in_map']
ingredient_name_to_rxcui = lookups['ingredient_name_to_rxcui']

print(f"✅ Loaded RxNorm mappings:")
print(f"   Name→RxCUI: {len(name_to_rxcui):,}")
print(f"   RxCUI→Names: {len(rxcui_to_names):,}")
print(f"   Brand→Ingredient: {len(bn_to_in_map):,}")
print(f"   Ingredient cache: {len(ingredient_name_to_rxcui):,}")

✅ Loaded RxNorm mappings:
   Name→RxCUI: 157,972
   RxCUI→Names: 82,134
   Brand→Ingredient: 77,518
   Ingredient cache: 6,409
